## Setup

In [4]:
import os, numpy as np, torch as t
import importlib, pipeline, model
importlib.reload(pipeline); importlib.reload(model)

from model    import Config
from pipeline import OptimizerSpec, grid_search, aggregate_grid, print_grid_table

t.manual_seed(0); np.random.seed(0)


BASE_CONFIG_15K = Config(
    p=113, d_model=128, d_mlp=512, num_heads=4, n_ctx=3,
    act_type='ReLU', frac_train=0.3,
    num_epochs=15_000,
    seed=0,
)
BASE_CONFIG_25K = Config(
    p=113, d_model=128, d_mlp=512, num_heads=4, n_ctx=3,
    act_type='ReLU', frac_train=0.3,
    num_epochs=25_000,    # AdamW a besoin de plus d'epochs pour Nanda baseline
    seed=0,
)


FILTER_INTERNAL_2D = lambda n, p: p.ndim >= 2 and 'embed' not in n

SEEDS = [0, 1]

FOURIER_EVERY = None


## 1 — EGD hybrid (5 lr × 5 wd × 2 momentum = 50 combos)



In [5]:
def make_egd_hybrid(lr_egd, wd_egd, momentum_egd):
    return [
        OptimizerSpec(
            name='egd',
            lr=lr_egd,
            weight_decay=wd_egd,
            param_filter=FILTER_INTERNAL_2D,
            extra={'momentum': momentum_egd, 'mode': 'svd'},
        ),
        OptimizerSpec(
            name='adamw',
            lr=1e-3, weight_decay=1.0,
            extra={'betas': (0.9, 0.98)},
        ),
    ]

PARAM_GRID_EGD = {
    'lr_egd':       [1e-3, 1e-2, 1e-1, 0.3, 1.0],
    'wd_egd':       [0,    1e-3, 1e-2, 1e-1, 1.0],
    'momentum_egd': [0.0, 0.9],
}
SAVE_EGD = 'runs/grid_clean/egd_hybrid'
n = int(np.prod([len(v) for v in PARAM_GRID_EGD.values()]))
print(f'EGD : {n} combos × {len(SEEDS)} seeds = {n*len(SEEDS)} runs  →  {SAVE_EGD}/')

results_egd = grid_search(
    BASE_CONFIG_15K, make_egd_hybrid, PARAM_GRID_EGD,
    seeds=SEEDS, save_root=SAVE_EGD,
    num_epochs=BASE_CONFIG_15K.num_epochs,
    eval_every=50, fourier_every=FOURIER_EVERY,
    warmup_steps=10, verbose_every=2_000, verbose_build=False,
)
rows_egd = aggregate_grid(results_egd, param_keys=list(PARAM_GRID_EGD.keys()),
                          acc_thresh=0.99, robustness_thresh=0.5)
for m in PARAM_GRID_EGD['momentum_egd']:
    sub = [r for r in rows_egd if r['momentum_egd'] == m]
    print(f'\n=== EGD — momentum = {m} ===')
    print_grid_table(sub, param_keys=['lr_egd', 'wd_egd'])

EGD : 50 combos × 2 seeds = 100 runs  →  runs/grid_clean/egd_hybrid/

=== grid_search: 50 combinations × 2 seeds = 100 runs ===
   params : ['lr_egd', 'wd_egd', 'momentum_egd']
   save   : runs/grid_clean/egd_hybrid/

--- [1/50] lr_egd0.001__wd_egd0__momentum_egd0 ---
   already done : [0, 1]
--- [2/50] lr_egd0.001__wd_egd0__momentum_egd0.9 ---
   already done : [0, 1]
--- [3/50] lr_egd0.001__wd_egd0.001__momentum_egd0 ---
   already done : [0, 1]
--- [4/50] lr_egd0.001__wd_egd0.001__momentum_egd0.9 ---
   already done : [0, 1]
--- [5/50] lr_egd0.001__wd_egd0.01__momentum_egd0 ---
   already done : [0, 1]
--- [6/50] lr_egd0.001__wd_egd0.01__momentum_egd0.9 ---
   already done : [0, 1]
--- [7/50] lr_egd0.001__wd_egd0.1__momentum_egd0 ---
   already done : [0, 1]
--- [8/50] lr_egd0.001__wd_egd0.1__momentum_egd0.9 ---
   already done : [0, 1]
--- [9/50] lr_egd0.001__wd_egd1__momentum_egd0 ---
   already done : [0, 1]
--- [10/50] lr_egd0.001__wd_egd1__momentum_egd0.9 ---
   already done : 

## 2 — Muon hybrid (momentum = 0.95) — 5 lr × 5 wd = 25 combos

In [ ]:
def make_muon_hybrid(lr_muon, wd_muon):
    return [
        OptimizerSpec(
            name='muon',
            lr=lr_muon,
            weight_decay=wd_muon,
            param_filter=FILTER_INTERNAL_2D,
            extra={'momentum': 0.95},
        ),
        OptimizerSpec(
            name='adamw',
            lr=1e-3, weight_decay=1.0,
            extra={'betas': (0.9, 0.98)},
        ),
    ]

PARAM_GRID_MUON = {
    'lr_muon': [1e-3, 1e-2, 1e-1, 0.3, 1.0],
    'wd_muon': [0,    1e-3, 1e-2, 1e-1, 1.0],
}
SAVE_MUON = 'runs/grid_clean/muon_hybrid'
n = int(np.prod([len(v) for v in PARAM_GRID_MUON.values()]))
print(f'Muon (m=0.95) : {n} combos × {len(SEEDS)} seeds = {n*len(SEEDS)} runs  →  {SAVE_MUON}/')

results_muon = grid_search(
    BASE_CONFIG_15K, make_muon_hybrid, PARAM_GRID_MUON,
    seeds=SEEDS, save_root=SAVE_MUON,
    num_epochs=BASE_CONFIG_15K.num_epochs,
    eval_every=50, fourier_every=FOURIER_EVERY,
    warmup_steps=10, verbose_every=2_000, verbose_build=False,
)
rows_muon = aggregate_grid(results_muon, param_keys=list(PARAM_GRID_MUON.keys()),
                           acc_thresh=0.99, robustness_thresh=0.5)
print('\n=== Muon hybrid (m=0.95) ===')
print_grid_table(rows_muon, param_keys=list(PARAM_GRID_MUON.keys()))

## 3 — Muon hybrid (momentum = 0, ablation) — 5 lr × 5 wd = 25 combos


In [ ]:
def make_muon_no_mom(lr_muon, wd_muon):
    return [
        OptimizerSpec(
            name='muon',
            lr=lr_muon,
            weight_decay=wd_muon,
            param_filter=FILTER_INTERNAL_2D,
            extra={'momentum': 0.0},   # ← ablation
        ),
        OptimizerSpec(
            name='adamw',
            lr=1e-3, weight_decay=1.0,
            extra={'betas': (0.9, 0.98)},
        ),
    ]

# même grille que muon_hybrid
SAVE_MUON_NM = 'runs/grid_clean/muon_no_mom'
n = int(np.prod([len(v) for v in PARAM_GRID_MUON.values()]))
print(f'Muon (m=0) : {n} combos × {len(SEEDS)} seeds = {n*len(SEEDS)} runs  →  {SAVE_MUON_NM}/')

results_muon_nm = grid_search(
    BASE_CONFIG_15K, make_muon_no_mom, PARAM_GRID_MUON,
    seeds=SEEDS, save_root=SAVE_MUON_NM,
    num_epochs=BASE_CONFIG_15K.num_epochs,
    eval_every=50, fourier_every=FOURIER_EVERY,
    warmup_steps=10, verbose_every=2_000, verbose_build=False,
)
rows_muon_nm = aggregate_grid(results_muon_nm, param_keys=list(PARAM_GRID_MUON.keys()),
                              acc_thresh=0.99, robustness_thresh=0.5)
print('\n=== Muon (m=0, ablation) ===')
print_grid_table(rows_muon_nm, param_keys=list(PARAM_GRID_MUON.keys()))

## 4 — AdamW pure — 5 lr × 5 wd = 25 combos (num_epochs = **25k**)


In [ ]:
# IMPORTANT : les keys de la grille sont 'lr' et 'weight_decay' (pas 'lr_adamw'/'wd_adamw')
# pour matcher le naming des dirs déjà existants dans grid_clean/adamw/.
def make_adamw_pure(lr, weight_decay):
    return [
        OptimizerSpec(
            name='adamw',
            lr=lr,
            weight_decay=weight_decay,
            extra={'betas': (0.9, 0.98)},
        ),
    ]

PARAM_GRID_ADAMW = {
    'lr':           [1e-4, 1e-3, 1e-2, 3e-2, 1e-1],
    'weight_decay': [0,    1e-2, 1e-1, 1.0,  10.0],
}
SAVE_ADAMW = 'runs/grid_clean/adamw'
n = int(np.prod([len(v) for v in PARAM_GRID_ADAMW.values()]))
print(f'AdamW : {n} combos × {len(SEEDS)} seeds = {n*len(SEEDS)} runs  →  {SAVE_ADAMW}/')

results_adamw = grid_search(
    BASE_CONFIG_25K, make_adamw_pure, PARAM_GRID_ADAMW,
    seeds=SEEDS, save_root=SAVE_ADAMW,
    num_epochs=BASE_CONFIG_25K.num_epochs,
    eval_every=50, fourier_every=FOURIER_EVERY,
    warmup_steps=10, verbose_every=2_000, verbose_build=False,
)
rows_adamw = aggregate_grid(results_adamw, param_keys=list(PARAM_GRID_ADAMW.keys()),
                            acc_thresh=0.99, robustness_thresh=0.5)
print('\n=== AdamW pure ===')
print_grid_table(rows_adamw, param_keys=list(PARAM_GRID_ADAMW.keys()))

## 5 — Audit final : combos × seeds présents dans `runs/grid_clean/`

In [ ]:
from pathlib import Path
ROOT = Path('runs/grid_clean')
if not ROOT.exists():
    print(f'⚠ {ROOT} n\\'existe pas — lance au moins une cellule grid.'); 
else:
    for opt_dir in sorted(ROOT.iterdir()):
        if not opt_dir.is_dir(): continue
        combos = [d for d in opt_dir.iterdir() if d.is_dir()]
        seed_counts = []
        for c in combos:
            n = sum(1 for s in c.iterdir() if s.name.startswith('seed') and (s/'history.json').exists())
            seed_counts.append(n)
        if seed_counts:
            print(f'{opt_dir.name:14s} : {len(combos)} combos, '
                  f'seed range [{min(seed_counts)}–{max(seed_counts)}], '
                  f'total runs = {sum(seed_counts)}')